# Moving application of ShellSIM from 1D time series data to complex 4D gridded dataset

Needed Variables 

T_timeseries=  Temperature Time series  
S_timeseries=  Practical Salinity Time series  
Chl_timeseries= Chlorophyll Time series  
POC_timeseries= Particulate Organic Carbon Time series  
POM_timeseries=  Particulate Organic Matter Time series  
TPM_timeseries= Total Particulate Matter   Time series  

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import pyfabm
import os
from dask.diagnostics import ProgressBar
import warnings
import gc
import datetime
from contextlib import redirect_stdout # For redirecting stdout

In [2]:
# Define the integration time period 
start = '02-06-2021'
end = '04-06-2021'
time_horizon = pd.date_range(start=start, end=end, freq='1d')
time_horizon_len = len(time_horizon)

In [3]:
# #  Lat lon chunking method
# chunking_config={'time': -1, 'latitude': 80, 'longitude': 110}

# def load_nc_file(file_path, var_name_in_file, chunking_config=chunking_config):
#     """
#     Loads a NetCDF file or creates a fake one if not found.
#     Args:
#         file_path (str): Path to the .nc file.
#         var_name_in_file (str): The name of the variable to use.
#         chunking_config (dict): Dask chunking configuration.
#     Returns:
#         xr.Dataset
#     """
#     fake_filename = f'{var_name_in_file}_gridded_FAKE.nc'
#     if os.path.exists(file_path):
#         print(f"Successfully loaded: {var_name_in_file}\n")
#         return xr.open_dataset(file_path, chunks=chunking_config)

#     if os.path.exists(fake_filename):
#         print(f"Using existing fake dataset: {fake_filename}\n")
#         return xr.open_dataset(fake_filename, chunks=chunking_config)

#     print(f"Creating fake gridded data for variable '{var_name_in_file}'...")
#     fake_data = np.random.rand(time_horizon_len, 100, 100)
#     fake_lats = np.linspace(40, 50, 100)
#     fake_lons = np.linspace(-10, 0, 100)

#     fake_gridded_dataset = xr.Dataset(
#         {var_name_in_file: (['time', 'latitude', 'longitude'], fake_data)},
#         coords={'time': time_horizon, 'latitude': fake_lats, 'longitude': fake_lons}
#     )
#     fake_gridded_dataset.to_netcdf(fake_filename)
#     print(f"Saved fake dataset to: {fake_filename}\n")
#     return xr.open_dataset(fake_filename, chunks=chunking_config)
        

In [4]:
#  Lat lon chunking method
chunking_config={'time': -1, 'latitude': 80, 'longitude': 110}
# Earth's radius in kilometers (used for converting degrees to distance)
# The mean radius is a good approximation for surface area calculation.
EARTH_RADIUS_KM = 6371.0 # km
_GLOBAL_COORDS = None


def load_nc_file(file_path, var_name_in_file, chunking_config=None):
    """
    Loads a NetCDF file or creates a fake one if not found.
    """
    global _GLOBAL_COORDS
    fake_filename = f'{var_name_in_file}_gridded_FAKE.nc'
    
    if os.path.exists(file_path):
        print(f"Successfully loaded: {var_name_in_file}")
        # Load the dataset
        ds = xr.open_dataset(file_path, chunks=chunking_config)
        # --- START COORDINATE & AREA CALCULATION BLOCK ---
        try:
            # 1. Get Lat/Lon coordinates safely
            lat_coords = ds['latitude'].values if 'latitude' in ds.coords else ds['lat'].values
            lon_coords = ds['longitude'].values if 'longitude' in ds.coords else ds['lon'].values
    
            # Get time coordinate if it exists
            time_coords = ds['time'].values if 'time' in ds.coords else None
            
            # Store global coordinates from the FIRST real dataset loaded
            if _GLOBAL_COORDS is None:
                _GLOBAL_COORDS = {
                    'lat': lat_coords,
                    'lon': lon_coords,
                    'time': time_coords,  
                    'lat_name': 'latitude' if 'latitude' in ds.coords else 'lat',
                    'lon_name': 'longitude' if 'longitude' in ds.coords else 'lon'
                }
                print(f"✅ Set global coordinate reference from: {var_name_in_file}")
            
        except KeyError:
            print("❌ ERROR: The dataset is missing coordinate variables. Expected ('latitude' or 'lat') and ('longitude' or 'lon').")
            return ds

        # Calculate bounds and ranges
        lat_min, lat_max = lat_coords.min(), lat_coords.max()
        lon_min, lon_max = lon_coords.min(), lon_coords.max()

        lat_range_deg = lat_max - lat_min
        lon_range_deg = lon_max - lon_min
        
        # 2. Convert ranges to distance
        lat_distance_km = lat_range_deg * (2 * np.pi * EARTH_RADIUS_KM / 360)
        
        mid_lat_rad = np.deg2rad((lat_max + lat_min) / 2)
        lon_distance_km = lon_range_deg * (2 * np.pi * EARTH_RADIUS_KM / 360) * np.cos(mid_lat_rad)
        
        # 3. Calculate Approximate Area
        approx_area_sq_km = lat_distance_km * lon_distance_km
        
        print(f"🗺️ Geographic Coverage:")
        print(f"  BBOX (xMin, yMin, xMax, yMax)")
        print(f"  BBOX ({lon_min:.2f}, {lat_min:.2f}, {lon_max:.2f}, {lat_max:.2f})")
        print(f"  Lat Range: {lat_min:.2f}° to {lat_max:.2f}° ({lat_range_deg:.2f}°)")
        print(f"  Lon Range: {lon_min:.2f}° to {lon_max:.2f}° ({lon_range_deg:.2f}°)")
        print(f"  Approximate Area: {approx_area_sq_km:,.0f} km²")
        print("---------------------------------------------------\n")
        
        return ds
    
    if os.path.exists(fake_filename):
        print(f"Using existing fake dataset: {fake_filename}\n")
        return xr.open_dataset(fake_filename, chunks=chunking_config)

    # Check if we have global coordinates from a real dataset
    if _GLOBAL_COORDS is None:
        print("⚠️ WARNING: No real dataset loaded yet. Creating fake data with default coordinates.")
        # Default fallback coordinates (your original values)
        fake_lats = np.linspace(40, 50, 100)
        fake_lons = np.linspace(-10, 0, 100)
        time_horizon = np.arange(0, 30)  # Default 30 time steps
        lat_name = 'latitude'
        lon_name = 'longitude'
    else:
        print(f"📍 Creating fake data matching global coordinate reference")
        fake_lats = _GLOBAL_COORDS['lat']
        fake_lons = _GLOBAL_COORDS['lon']
        lat_name = _GLOBAL_COORDS['lat_name']
        lon_name = _GLOBAL_COORDS['lon_name']
        
        # Use time from real dataset if available, otherwise default
        if _GLOBAL_COORDS.get('time') is not None:  # ← CHANGED to .get()
            time_horizon = _GLOBAL_COORDS['time']
        else:
            time_horizon = np.arange(0, 30)
    
    time_horizon_len = len(time_horizon)
    
    # Create fake data with matching dimensions
    fake_data = np.random.rand(time_horizon_len, len(fake_lats), len(fake_lons))
    
    fake_gridded_dataset = xr.Dataset(
        {var_name_in_file: (['time', lat_name, lon_name], fake_data)},
        coords={'time': time_horizon, lat_name: fake_lats, lon_name: fake_lons}
    )
    
    fake_gridded_dataset.to_netcdf(fake_filename)
    print(f"✅ Saved fake dataset to: {fake_filename}")
    print(f"   Coordinates: time[{time_horizon_len}], {lat_name}[{len(fake_lats)}], {lon_name}[{len(fake_lons)}]")
    print(f"   BBOX ({fake_lons.min():.2f}, {fake_lats.min():.2f}, {fake_lons.max():.2f}, {fake_lats.max():.2f})\n")
    
    return xr.open_dataset(fake_filename, chunks=chunking_config)

    

In [5]:
# Define datasets to be read and used for model
poc_file_path="/home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_BIO_BGC_3D_REP_015_010/cmems_obs-mob_glo_bgc-chl-poc_my_0.25deg_P7D-m_poc_13.00W-42.00E_30.00N-70.00N_0.00-1000.00m_2021-06-01-2021-06-30_3f1f32d8adbe3d686e11d7d4a40be7bb.nc"
poc_var_name = 'poc'  # variable name inside poc.nc

salinity_file_path="/home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc"
salinity_var_name = 'sos' # variable name inside salinity.nc

# Fake data generation to be triggered in the 'except' block of load_nc_file if path doesn't exist
temp_file_path = 'non_existent_temp.nc'
temp_var_name = 'temperature' # variable name inside temperature.nc

chl_file_path = 'non_existent_chl.nc'
chl_var_name = 'chl'

pom_file_path = 'non_existent_pom.nc'
pom_var_name = 'pom'

tpm_file_path = 'non_existent_tpm.nc'
tpm_var_name = 'tpm'


In [6]:
# Load all datasets
print("Loading datasets...\n")
# view bbox with http://bboxfinder.com/#30.060000,-12.940000,69.940000,41.940000
ds_poc = load_nc_file(poc_file_path, poc_var_name)
ds_sal = load_nc_file(salinity_file_path, salinity_var_name)
ds_temp = load_nc_file(temp_file_path, temp_var_name)
ds_chl = load_nc_file(chl_file_path, chl_var_name)
ds_pom = load_nc_file(pom_file_path, pom_var_name)
ds_tpm = load_nc_file(tpm_file_path, tpm_var_name)

Loading datasets...

Successfully loaded: poc


getfattr: /home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_BIO_BGC_3D_REP_015_010/cmems_obs-mob_glo_bgc-chl-poc_my_0.25deg_P7D-m_poc_13.00W-42.00E_30.00N-70.00N_0.00-1000.00m_2021-06-01-2021-06-30_3f1f32d8adbe3d686e11d7d4a40be7bb.nc: Operation not supported
getfattr: /home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc: Operation not supported


✅ Set global coordinate reference from: poc
🗺️ Geographic Coverage:
  BBOX (xMin, yMin, xMax, yMax)
  BBOX (-12.88, 30.12, 41.88, 69.88)
  Lat Range: 30.12° to 69.88° (39.75°)
  Lon Range: -12.88° to 41.88° (54.75°)
  Approximate Area: 17,296,519 km²
---------------------------------------------------

Successfully loaded: sos
🗺️ Geographic Coverage:
  BBOX (xMin, yMin, xMax, yMax)
  BBOX (-12.94, 30.06, 41.94, 69.94)
  Lat Range: 30.06° to 69.94° (39.88°)
  Lon Range: -12.94° to 41.94° (54.88°)
  Approximate Area: 17,390,524 km²
---------------------------------------------------

Using existing fake dataset: temperature_gridded_FAKE.nc

Using existing fake dataset: chl_gridded_FAKE.nc

Using existing fake dataset: pom_gridded_FAKE.nc

Using existing fake dataset: tpm_gridded_FAKE.nc



In [7]:
# Align coordinates - using POC data as reference
# Grid alignment using ds_poc as the reference grid and interpolating all other variables to it, 
# as a way to handle mismatched grids.
print("Aligning coordinates...")
ref_lats = ds_poc.latitude
ref_lons = ds_poc.longitude


Aligning coordinates...


In [8]:
interp_kwargs = {'fill_value': 'extrapolate'}

# For 4D variables: Select first depth values -  Select a single depth level and interpolate 
# use .sel(depth=0, method='nearest') to grab the surface layer
poc_daily = ds_poc[poc_var_name].sel(depth=0, method='nearest').interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
sal_daily = ds_sal[salinity_var_name].sel(depth=0, method='nearest').interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)


# Process 3D variables (datasets with no depth)
temp_daily = ds_temp[temp_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
chl_daily = ds_chl[chl_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
pom_daily = ds_pom[pom_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
tpm_daily = ds_tpm[tpm_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)


# Merge into a single dataset
ds_daily = xr.Dataset({
    'salinity': sal_daily,
    'POC': poc_daily,
    'temperature': temp_daily,
    'Chl': chl_daily,
    'POM': pom_daily,
    'TPM': tpm_daily
})

print("Merged daily dataset")
ds_daily


Merged daily dataset


<xarray.Dataset> Size: 101MB
Dimensions:      (time: 60, latitude: 160, longitude: 220)
Coordinates:
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
  * latitude     (latitude) float32 640B 30.12 30.38 30.62 ... 69.38 69.62 69.88
  * longitude    (longitude) float32 880B -12.88 -12.62 -12.38 ... 41.62 41.88
    depth        float32 4B 0.0
Data variables:
    salinity     (time, latitude, longitude) float64 17MB 39.33 38.18 ... 22.59
    POC          (time, latitude, longitude) float64 17MB nan nan ... nan nan
    temperature  (time, latitude, longitude) float64 17MB -0.8622 ... 0.1612
    Chl          (time, latitude, longitude) float64 17MB 9.507 ... -3.106
    POM          (time, latitude, longitude) float64 17MB 10.51 ... -2.952
    TPM          (time, latitude, longitude) float64 17MB 2.039 ... -5.684

In [9]:
def subset_ds(ds: xr.Dataset, bbox: tuple) -> xr.Dataset:
    """
    Subsets an xarray.Dataset to a given geographic bounding box.
    The function assumes the dataset has 'latitude' and 'longitude'
    coordinates.
    Args:
        ds (xr.Dataset): The dataset to subset.
        bbox (tuple): A tuple containing the bounding box in the
                      format (min_lon, min_lat, max_lon, max_lat).
    Returns:
        xr.Dataset: The spatially subsetted dataset.
    """
    # Unpack the bounding box
    min_lon, min_lat, max_lon, max_lat = bbox

    # Use the .sel() method with slice() to select the data
    # within the bounding box. This is the standard xarray way
    # to select a range of coordinate values.
    subset = ds.sel(
        latitude=slice(min_lat, max_lat),
        longitude=slice(min_lon, max_lon)
    )

    return subset

In [10]:
# subset entire dataset
# 7.910156,53.041213,31.025391,69.364831
# http://bboxfinder.com/#53.041213,7.910156,69.364831,31.025391
bbox = (7.910156, 53.041213, 31.025391, 69.364831)

ds_daily = subset_ds(ds_daily, bbox)
ds_daily

<xarray.Dataset> Size: 17MB
Dimensions:      (time: 60, latitude: 65, longitude: 92)
Coordinates:
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
  * latitude     (latitude) float32 260B 53.12 53.38 53.62 ... 68.62 68.88 69.12
  * longitude    (longitude) float32 368B 8.125 8.375 8.625 ... 30.62 30.88
    depth        float32 4B 0.0
Data variables:
    salinity     (time, latitude, longitude) float64 3MB nan nan nan ... nan nan
    POC          (time, latitude, longitude) float64 3MB nan nan nan ... nan nan
    temperature  (time, latitude, longitude) float64 3MB 10.05 3.67 ... 0.1293
    Chl          (time, latitude, longitude) float64 3MB 6.55 -5.237 ... 5.875
    POM          (time, latitude, longitude) float64 3MB 4.772 3.885 ... 4.057
    TPM          (time, latitude, longitude) float64 3MB -15.31 ... 0.2519

In [11]:
# del ds_poc  # Free memory immediately if needed 
# del ds_sal
# del ds_temp
# del ds_chl
# del ds_pom
# del ds_tpm
# gc.collect()

In [12]:
# Rechunk for optimal performance
# After all the merging and interpolating, Dask's chunks can get fragmented, rechunk so every Dask task receives 
# a data chunk of the exact size

print("Rechunking dataset for optimal performance")
ds_daily = ds_daily.chunk({
    'time': -1,       # Keep all time steps together
    'latitude': 80,   # Process 80 latitudes at once
    'longitude': 110  # Process 110 longitudes at once
})
print("Rechunked dataset")
print(ds_daily.chunks)

Rechunking dataset for optimal performance
Rechunked dataset
Frozen({'time': (60,), 'latitude': (65,), 'longitude': (92,)})


# ShellSIM model wraapper   
takes a 1D numpy array (time-series for one pixel) as input. run entire for loop (time-stepping) and return a 1D numpy array of the result ( eg soft tissue energy time-series). Then Apply in Parallel, using xr.apply_ufunc to apply wrapper function to the gridded data. tell apply_ufunc that the "core dimension" is time, which instructs it to parallelize over all other dimensions (lat, lon)

In [13]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_LOG_FILENAME = f"fabm_run_log_{timestamp}.log"


def run_fabm_at_point_full(T_ts, S_ts, Chl_ts, POC_ts, POM_ts, TPM_ts, log_filename):
    """
    Runs FABM time-loop for a single spatial point.
    Returns a 1D array of output, e.g., Soft Tissue Energy.
    """
    # Check for NaN inputs
    if (np.any(np.isnan(T_ts)) or np.any(np.isnan(S_ts)) or 
        np.any(np.isnan(Chl_ts)) or np.any(np.isnan(POC_ts)) or 
        np.any(np.isnan(POM_ts)) or np.any(np.isnan(TPM_ts))):
        return np.full(time_horizon_len, np.nan)
    
    # Check for non-finite values
    if not (np.all(np.isfinite(T_ts)) and np.all(np.isfinite(S_ts)) and
            np.all(np.isfinite(Chl_ts)) and np.all(np.isfinite(POC_ts)) and
            np.all(np.isfinite(POM_ts)) and np.all(np.isfinite(TPM_ts))):
        return np.full(time_horizon_len, np.nan)
    
    try:
        # --- START OUTPUT REDIRECTION BLOCK ---
        # with open(os.devnull, 'w') as fnull:
        #     with redirect_stdout(fnull):
        with open(log_filename, 'a') as f_log:
            with redirect_stdout(f_log):
                
                # Initialize model
                model = pyfabm.Model("/home/jovyan/work/ShellSIM_Trials/notebook_timeseries/fabm.yaml")
                
                # Set static dependencies
                model.cell_thickness = 1.0
                model.dependencies["seeding_rate"].value = 0.0
                model.dependencies["harvest_ratio"].value = 0.0
                model.dependencies["current_speed"].value = 1.0
                model.dependencies["air_exposure"].value = 0.0
                model.dependencies["number_of_days_since_start_of_the_year"].value = 0.0
        
        
                # Set INITIAL values for all dependencies
                # We use the first day's value (index 0) to initialize.
                model.dependencies["temperature"].value = float(T_ts[0])
                model.dependencies["practical_salinity"].value = float(S_ts[0])
                model.findStateVariable('Chl1/Chl').value = float(Chl_ts[0])
                model.findStateVariable('POC1/POC').value = float(POC_ts[0])
                model.findStateVariable('POM1/POM').value = float(POM_ts[0])
                model.findStateVariable('TPM1/TPM').value = float(TPM_ts[0])
        
                if not model.start():
                    raise RuntimeError("FABM model failed to start internally.")
        
        
        # Initialize output
        outputs = np.zeros(time_horizon_len)
        
        
        # Daily integration
        for nd in range(time_horizon_len):
            # Set environmental variables
            model.dependencies["temperature"].value = float(T_ts[nd])
            model.dependencies["practical_salinity"].value = float(S_ts[nd])
            
            # Set food variables
            model.findStateVariable('Chl1/Chl').value = float(Chl_ts[nd])
            model.findStateVariable('POC1/POC').value = float(POC_ts[nd])
            model.findStateVariable('POM1/POM').value = float(POM_ts[nd])
            model.findStateVariable('TPM1/TPM').value = float(TPM_ts[nd])
            
            # Calculate and apply growth rates
            state_rates = model.getRates()

            # Perfom forward Euler integration
            model.state[:] += state_rates * 86400.
            
            # Save output (soft_tissue_output)
            outputs[nd] = model.state[0]

        #      # Save all 11 states
        #     outputs[:, nd] = model.state[:] 
        # # RETURN THE FULL 2D OUTPUT ARRAY
        # return outputs
        return outputs

    except RuntimeError as e: # Catch the internal failure we raised
        warnings.warn("!!! Model failed to start even after setting initial dependencies.")
        # We can still print the specific FABM error message outside the redirection
        print(f"Model failed to start: {pyfabm.getError()}")
        return np.full(time_horizon_len, np.nan)
        
    except Exception as e:
        warnings.warn(f"FABM error: {str(e)}")
        return np.full(time_horizon_len, np.nan)

## Most optimal processing method:  parallelize over the spatial dimensions (latitude, longitude) using xarray.apply_ufunc.  
chunking only the spatial dimensions (latitude and longitude) tells Dask to split the map into tiles, but keep the full time series for each pixel intact.

In [14]:
# apply_ufunc call. It accepts 6 distinct numpy arrays, one for each variable's time series at a single point.
# passes the 6 DataArrays from ds_daily to the 6 arguments of run_fabm_at_point_full
# Apply the model across the grid
# xarray.apply_ufunc: the clean way to run a function over each pixel (lat/lon) in parallel, using Dask underneath
# Dask will distribute this across threads or processes as needed.

print("Setting up parallel computation with apply_ufunc ...")

# result_sten = soft tissue energy 
result_sten = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # input data arrays
    ds_daily['temperature'],   
    ds_daily['salinity'],
    ds_daily['Chl'],
    ds_daily['POC'],
    ds_daily['POM'],
    ds_daily['TPM'],

    # function consumes 6 1D time array
    input_core_dims=[['time']] * 6,
    # outputs also with  a time dimension
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    # use Dask parallel execution
    dask='parallelized',
    # run over all spatial dimensions
    vectorize=True,
    output_dtypes=[float],
    kwargs={'log_filename': RUN_LOG_FILENAME},
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)


# Add time coordinate back and set attributes
result_sten = result_sten.assign_coords(time=time_horizon)

# output rename for clarity 
result_sten = result_sten.rename('soft_tissue_energy')
result_sten.attrs['units'] = 'J'
result_sten.attrs['long_name'] = 'Oyster Soft Tissue Energy'

print("\nTask graph built. Ready to compute")
print(result_sten)

Setting up parallel computation with apply_ufunc ...

Task graph built. Ready to compute
<xarray.DataArray 'soft_tissue_energy' (latitude: 65, longitude: 92, time: 60)> Size: 3MB
dask.array<transpose, shape=(65, 92, 60), dtype=float64, chunksize=(65, 92, 60), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float32 260B 53.12 53.38 53.62 ... 68.62 68.88 69.12
  * longitude  (longitude) float32 368B 8.125 8.375 8.625 ... 30.38 30.62 30.88
  * time       (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
    depth      float32 4B 0.0
Attributes:
    units:      J
    long_name:  Oyster Soft Tissue Energy


# Run computation and Save  
Call .compute() or .to_netcdf() on the result. This triggers Dask to execute the parallel computation and write the final 3D output file.

In [15]:
# Compute with progress bar and save output when done
output_file_name = "direct_gridded_oyster_output.nc" 

print("Now running dask computation ...")
with ProgressBar():
    result_sten.to_netcdf(output_file_name, compute=True)

print(f" ****** SUCCESS: Results saved to {output_file_name} ******* ")

Now running dask computation ...
[########################################] | 100% Completed | 1.23 sms
 ****** SUCCESS: Results saved to direct_gridded_oyster_output.nc ******* 


In [16]:
# # Clean up
# del ds_poc  
# del ds_sal
# del ds_temp
# del ds_chl
# del ds_pom
# del ds_tpm
# del ds_daily, result_sten
# gc.collect()

# Method 2 on Memory optimization    
Runs in 1:37 ish for start = '02-06-2021' end = '04-06-2021'

In [17]:
time_horizon_len = ds_daily.time.size 
output_file_name = "gridded_oyster_output_soft_tissue_energy_batched.nc"


# 2. BATCHING LOOP
# Use  chunk sizes of  data
# Extract chunk definitions from the input data
# This assumes your ds_daily is already chunked (e.g., in Cell 3 of the original notebook)
try:
    lat_chunks = ds_daily.chunks['latitude']
    lon_chunks = ds_daily.chunks['longitude']
except KeyError:
    print("Error: ds_daily must be a Dask-backed xarray object with defined 'latitude' and 'longitude' chunks.")
    # Exit or provide placeholder logic if necessary
    raise

lat_indices = np.cumsum([0] + list(lat_chunks))
lon_indices = np.cumsum([0] + list(lon_chunks))

print(f"Starting batched computation and saving to {output_file_name}...")

# Loop over Latitude chunks
for i in range(len(lat_chunks)):
    lat_start = lat_indices[i]
    lat_end = lat_indices[i+1]
    
    # Loop over Longitude chunks
    for j in range(len(lon_chunks)):
        lon_start = lon_indices[j]
        lon_end = lon_indices[j+1]

        # Select the spatial subset (chunk) for computation
        ds_subset = ds_daily.isel(
            latitude=slice(lat_start, lat_end),
            longitude=slice(lon_start, lon_end)
        )
        
        print(f"Processing batch: Lat {i+1}/{len(lat_chunks)}, Lon {j+1}/{len(lon_chunks)}")
        
        # --- Run the user's original apply_ufunc logic on the subset ---
        result_sten_batch = xr.apply_ufunc(
            run_fabm_at_point_full,
            
            # input data arrays (subset)
            ds_subset['temperature'],   
            ds_subset['salinity'],
            ds_subset['Chl'],
            ds_subset['POC'],
            ds_subset['POM'],
            ds_subset['TPM'],

            input_core_dims=[['time']] * 6,
            output_core_dims=[['time']],
            exclude_dims=set(('time',)),
            dask='parallelized',
            vectorize=True, 
            output_dtypes=[float],
            kwargs={'log_filename': RUN_LOG_FILENAME},
            dask_gufunc_kwargs={
                'allow_rechunk': True,
                'output_sizes': {'time': time_horizon_len}
            }
        )
        
        # Add time coordinate back and set attributes
        result_sten_batch = result_sten_batch.assign_coords(time=ds_subset.time)
        result_sten_batch = result_sten_batch.rename('soft_tissue_energy')
        result_sten_batch.attrs['units'] = 'J'
        result_sten_batch.attrs['long_name'] = 'Oyster Soft Tissue Energy'
        
        # --- Compute and Save (Force memory release for this batch) ---
        with ProgressBar():
            # Use 'w' (write) for the very first chunk (i=0, j=0) to create the file.
            # Use 'a' (append) for all subsequent chunks to add to the existing file.
            mode = 'w' if (i == 0 and j == 0) else 'a'
            
            # Convert to Dataset for cleaner NetCDF output/appending
            result_sten_batch.to_dataset().to_netcdf(
                output_file_name, 
                mode=mode, 
                format='NETCDF4', 
                compute=True
            )
            
print(f"✅ Success: All spatial batches processed and results saved incrementally to {output_file_name}")

Starting batched computation and saving to gridded_oyster_output_soft_tissue_energy_batched.nc...
Processing batch: Lat 1/1, Lon 1/1
[########################################] | 100% Completed | 1.17 sms
✅ Success: All spatial batches processed and results saved incrementally to gridded_oyster_output_soft_tissue_energy_batched.nc


## Read and explore produced output

In [18]:
output_file_path="/home/jovyan/work/ShellSIM_Trials/notebook_timeseries/gridded_oyster_output_soft_tissue_energy_batched.nc"
output_var = 'soft_tissue_energy'

ds_output = load_nc_file(output_file_path, output_var)
ds_output

Successfully loaded: soft_tissue_energy
🗺️ Geographic Coverage:
  BBOX (xMin, yMin, xMax, yMax)
  BBOX (8.12, 53.12, 30.88, 69.12)
  Lat Range: 53.12° to 69.12° (16.00°)
  Lon Range: 8.12° to 30.88° (22.75°)
  Approximate Area: 2,173,346 km²
---------------------------------------------------



<xarray.Dataset> Size: 3MB
Dimensions:             (latitude: 65, longitude: 92, time: 60)
Coordinates:
  * latitude            (latitude) float32 260B 53.12 53.38 ... 68.88 69.12
  * longitude           (longitude) float32 368B 8.125 8.375 ... 30.62 30.88
  * time                (time) datetime64[ns] 480B 2021-02-06 ... 2021-04-06
    depth               float32 4B ...
Data variables:
    soft_tissue_energy  (latitude, longitude, time) float64 3MB ...

In [19]:
output_file_path="/home/jovyan/work/ShellSIM_Trials/notebook_timeseries/direct_gridded_oyster_output.nc"
output_var = 'soft_tissue_energy'

ds_output = load_nc_file(output_file_path, output_var)
ds_output

Successfully loaded: soft_tissue_energy
🗺️ Geographic Coverage:
  BBOX (xMin, yMin, xMax, yMax)
  BBOX (8.12, 53.12, 30.88, 69.12)
  Lat Range: 53.12° to 69.12° (16.00°)
  Lon Range: 8.12° to 30.88° (22.75°)
  Approximate Area: 2,173,346 km²
---------------------------------------------------



<xarray.Dataset> Size: 3MB
Dimensions:             (latitude: 65, longitude: 92, time: 60)
Coordinates:
  * latitude            (latitude) float32 260B 53.12 53.38 ... 68.88 69.12
  * longitude           (longitude) float32 368B 8.125 8.375 ... 30.62 30.88
  * time                (time) datetime64[ns] 480B 2021-02-06 ... 2021-04-06
    depth               float32 4B ...
Data variables:
    soft_tissue_energy  (latitude, longitude, time) float64 3MB ...

In [20]:
# output_file_path="/home/jovyan/work/ShellSIM_Trials/notebook_timeseries/gridded_oyster_output_subsetpoint.nc"
# output_var = 'soft_tissue_energy'


# ds_output = load_nc_file(output_file_path, output_var)
# ds_output

In [21]:
# # Getting all 11 states 
# N_STATES = 11
# STATE_NAMES = ['soft_tissue_energy', 'shell_energy', 'aging', 'C1', 'C2', 'C3', 'Chl_state', 'POC_state', 'POM_state', 'TPM_state', 'O2']

# # Apply the model across the grid
# result_full = xr.apply_ufunc(
#     run_pyfabm_model,
#     # --- INPUT data ---
#     ds_daily['temperature'], 
#     ds_daily['salinity'],
#     ds_daily['Chl'],
#     ds_daily['POC'],
#     ds_daily['POM'],
#     ds_daily['TPM'],

#     # Function now returns a 2D array (n_states, n_time_steps)
#     input_core_dims=[['time']] * 6,
#     output_core_dims=[['state'], ['time']], # New 'state' dimension
    
#     exclude_dims=set(('time',)),
#     dask='parallelized',
#     output_dtypes=[float],
#     dask_gufunc_kwargs={
#         'allow_rechunk': True,
#         # Define the size for the new 'state' dimension
#         'output_sizes': {'state': N_STATES, 'time': time_horizon_len}
#     }
# )

# # Rename the dimensions and coordinates
# result_full = result_full.rename({'state': 'state_variable', 'time': 'time'})
# result_full = result_full.assign_coords(state_variable=STATE_NAMES)
# result_full = result_full.assign_coords(time=time_horizon)
# result_full = result_full.to_dataset(dim='state_variable')

# print("Task graph built for all 11 states. Ready to compute")
# print(result_full)

# Validate on single pixel or tiny grid 

In [22]:

def check_subset_validity(ds_list, var_names, lat, lon, depth=0):
    """
    Checks if a single nearest grid point exists and contains valid data (not all NaNs) for the specified variables and depth.

    Args:
        ds_list (list): List of xarray Datasets (e.g., [ds_poc, ds_sal]).
        var_names (list): List of variable names corresponding to ds_list (e.g., ['poc', 'salinity']).
        lat (float): Latitude of the validation point.
        lon (float): Longitude of the validation point.
        depth (float): Depth level to check (e.g., 0.0 for surface).
    """
    print(f"Checking Validity for Lat: {lat}, Lon: {lon}")
    
    # Select the nearest point (and surface depth) from all datasets
    ds_point = ds_list[0].sel(
        latitude=lat, 
        longitude=lon, 
        depth=depth,
        method='nearest'
    )
    
    # Check if a point was even found (i.e., not an empty slice)
    if 'latitude' in ds_point.coords:
        found_lat = ds_point.latitude.item()
        found_lon = ds_point.longitude.item()
        print(f"Nearest grid point found at: Lat={found_lat:.2f}, Lon={found_lon:.2f}")
    else:
        print(f"⚠️WARNING: No grid point found near {lat}, {lon}. Check your grid bounds.")
        return

    all_valid = True
    
    for ds, var_name in zip(ds_list, var_names):
        # Select the specific variable and the single point
        # Need to re-select the variable from its own source file
        try:
            data_var = ds[var_name].sel(
                latitude=lat, 
                longitude=lon, 
                depth=depth,
                method='nearest'
            )
            # Load the 1D time series and check for all NaNs
            is_all_nan = np.isnan(data_var.compute()).all()
            
            if is_all_nan:
                print(f"⚠️WARNING: {var_name} is ALL NaN at this location. No model run possible.")
                all_valid = False
            else:
                print(f"✅ {var_name} contains valid data.")
        except Exception as e:
            print(f"❌ERROR: checking {var_name}: {e}")
            all_valid = False

    if all_valid:
        print("✅SUCCESS: All critical variables contain valid data at this location. Ready to run the model!")
    else:
        print("❌FAILED: One or more critical variables are missing data. Try a new location.")

In [23]:
def check_subset_validity_return_nearest(ds_list, var_names, lat, lon, depth=0, search_radius_deg=2.0):
    """
    Checks if a single nearest grid point exists and contains valid data for all specified variables.
    If the nearest point has missing data, searches for the closest valid point within a search radius.

    Args:
        ds_list (list): List of xarray Datasets (e.g., [ds_poc, ds_sal]).
        var_names (list): List of variable names corresponding to ds_list (e.g., ['poc', 'salinity']).
        lat (float): Latitude of the validation point.
        lon (float): Longitude of the validation point.
        depth (float): Depth level to check (e.g., 0.0 for surface).
        search_radius_deg (float): Search radius in degrees to find valid points.

    Returns:
        tuple: ('passed', lat, lon) if valid data found, or (None, None, None) if no valid point found.
    """
    print(f"Checking Validity for Lat: {lat}, Lon: {lon}")
    
    # First, try the exact nearest point
    def check_point(check_lat, check_lon, verbose=False):
        """Helper function to check if a specific point has valid data for all variables."""
        try:
            ds_point = ds_list[0].sel(
                latitude=check_lat, 
                longitude=check_lon, 
                depth=depth,
                method='nearest'
            )
            
            if 'latitude' not in ds_point.coords:
                return None, None, False, []
            
            found_lat = ds_point.latitude.item()
            found_lon = ds_point.longitude.item()
            
            all_valid = True
            invalid_vars = []
            
            for ds, var_name in zip(ds_list, var_names):
                try:
                    data_var = ds[var_name].sel(
                        latitude=check_lat, 
                        longitude=check_lon, 
                        depth=depth,
                        method='nearest'
                    )
                    is_all_nan = np.isnan(data_var.compute()).all()
                    
                    if is_all_nan:
                        all_valid = False
                        invalid_vars.append(var_name)
                        if verbose:
                            print(f"  ❌ {var_name} is ALL NaN at this location")
                    else:
                        if verbose:
                            print(f"  ✅ {var_name} contains valid data")
                except Exception as e:
                    all_valid = False
                    invalid_vars.append(var_name)
                    if verbose:
                        print(f"  ❌ {var_name} error: {e}")
            
            return found_lat, found_lon, all_valid, invalid_vars
        except Exception as e:
            if verbose:
                print(f"  ❌ Error checking point: {e}")
            return None, None, False, var_names
    
    # Check the requested point first
    found_lat, found_lon, is_valid, invalid_vars = check_point(lat, lon, verbose=True)
    
    if found_lat is not None:
        print(f"Nearest grid point to request: Lat={found_lat:.2f}, Lon={found_lon:.2f}")
        
        if is_valid:
            print("✅SUCCESS: All variables contain valid data at this location!")
            return 'passed', found_lat, found_lon
        else:
            print(f"⚠️ Requested point has missing data for: {', '.join(invalid_vars)}")
            print("   Searching for nearest valid point...")
    else:
        print(f"⚠️WARNING: No grid point found near {lat}, {lon}.")
        return None, None, None
    
    # Search for a valid point in the surrounding area
    lat_min = lat - search_radius_deg
    lat_max = lat + search_radius_deg
    lon_min = lon - search_radius_deg
    lon_max = lon + search_radius_deg
    
    try:
        # Get a subset of the first dataset to get available lat/lon points
        ds_subset = ds_list[0].sel(
            latitude=slice(lat_min, lat_max),
            longitude=slice(lon_min, lon_max),
            depth=depth
        )
        
        # Get all lat/lon combinations in the subset
        lats = ds_subset.latitude.values
        lons = ds_subset.longitude.values
        
        print(f"   Searching {len(lats)} x {len(lons)} = {len(lats) * len(lons)} grid points...")
        
        # Calculate distances from the requested point
        best_distance = float('inf')
        best_lat = None
        best_lon = None
        points_checked = 0
        
        for test_lat in lats:
            for test_lon in lons:
                points_checked += 1
                # Calculate approximate distance (simple Euclidean for small areas)
                distance = np.sqrt((test_lat - lat)**2 + (test_lon - lon)**2)
                
                # Check if this point has valid data for all variables
                _, _, is_valid, _ = check_point(test_lat, test_lon, verbose=False)
                
                if is_valid and distance < best_distance:
                    best_distance = distance
                    best_lat = test_lat
                    best_lon = test_lon
        
        print(f"   Checked {points_checked} points in search area")
        
        if best_lat is not None:
            print(f"✅FOUND: Nearest valid point at Lat={best_lat:.2f}, Lon={best_lon:.2f}")
            print(f"  Distance from requested point: {best_distance:.3f} degrees (~{best_distance * 111:.1f} km)")
            
            # Print status for each variable at the found point
            for ds, var_name in zip(ds_list, var_names):
                print(f"  ✅ {var_name} contains valid data")
            
            return 'passed', best_lat, best_lon
        else:
            print(f"❌FAILED: No valid point found within {search_radius_deg} degrees of requested location.")
            print(f"  All {points_checked} points checked had missing data for one or more variables:")
            print(f"  Missing variables at requested point: {', '.join(invalid_vars)}")
            return None, None, None
    
    except Exception as e:
        print(f"❌ERROR during search: {e}")
        return None, None, None



# Validate if a given point exist in input variable data sources 

In [24]:


# # Coordinates for Gullmarsfjord, Sweden
# validation_lat = 58.2565
# validation_lon = 11.4444

# coordinates picked from the sample data source 
validation_lat = 45.125 
validation_lon = -7.875

# Run the check function
check_subset_validity(
    ds_list=[ds_poc, ds_sal], 
    var_names=[poc_var_name, salinity_var_name], 
    lat=validation_lat, 
    lon=validation_lon
)

Checking Validity for Lat: 45.125, Lon: -7.875
Nearest grid point found at: Lat=45.12, Lon=-7.88
✅ poc contains valid data.
✅ sos contains valid data.
✅SUCCESS: All critical variables contain valid data at this location. Ready to run the model!


In [25]:

# # Coordinates for Gullmarsfjord, Sweden
# validation_lat = 58.2565
# validation_lon = 11.4444

validation_lat = 45.125 
validation_lon = -7.875

print("Selecting single nearest grid point...")

# Select the single nearest grid point from the full dataset
ds_subset = ds_daily.sel(
    latitude=validation_lat, 
    longitude=validation_lon, 
    method='nearest'
)

# Load the single point data into memory
ds_subset.load()

print(" Validation point ready")
print(ds_subset)

Selecting single nearest grid point...
 Validation point ready
<xarray.Dataset> Size: 3kB
Dimensions:      (time: 60)
Coordinates:
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
    depth        float32 4B 0.0
    latitude     float32 4B 53.12
    longitude    float32 4B 8.125
Data variables:
    salinity     (time) float64 480B nan nan nan nan nan ... nan nan nan nan nan
    POC          (time) float64 480B nan nan nan nan nan ... nan nan nan nan nan
    temperature  (time) float64 480B 10.05 9.968 9.887 ... 5.403 5.321 5.24
    Chl          (time) float64 480B 6.55 6.497 6.444 ... 3.519 3.466 3.412
    POM          (time) float64 480B 4.772 4.737 4.701 ... 2.751 2.715 2.68
    TPM          (time) float64 480B -15.31 -15.18 -15.04 ... -7.645 -7.513


In [26]:

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_LOG_FILENAME = f"fabm_run_log_{timestamp}.log"

# Run apply_ufunc on  Subset
result_sten_subset = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # ---  SUBSET data ---
    ds_subset['temperature'],   
    ds_subset['salinity'],
    ds_subset['Chl'],
    ds_subset['POC'],
    ds_subset['POM'],
    ds_subset['TPM'],

    input_core_dims=[['time']] * 6,
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    dask='parallelized',
    output_dtypes=[float],
    kwargs={'log_filename': RUN_LOG_FILENAME},
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)


# Add time coordinate back and set attributes
result_sten_subset = result_sten_subset.assign_coords(time=time_horizon)
result_sten_subset = result_sten_subset.rename('soft_tissue_energy')
result_sten_subset.attrs['units'] = 'J'
result_sten_subset.attrs['long_name'] = 'Oyster Soft Tissue Energy'

print("Task graph built. Ready to compute subset.")
print(result_sten_subset)

Task graph built. Ready to compute subset.
<xarray.DataArray 'soft_tissue_energy' (time: 60)> Size: 480B
array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan])
Coordinates:
  * time       (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
    depth      float32 4B 0.0
    latitude   float32 4B 53.12
    longitude  float32 4B 8.125
Attributes:
    units:      J
    long_name:  Oyster Soft Tissue Energy


In [27]:

# Compute with progress bar
print(" Running dask computation on subset...")

# output_file_name = "gridded_oyster_output_GULLMARSFJORD.nc"
output_file_name = "gridded_oyster_output_subsetpoint.nc"
with ProgressBar():
    # Save to a new file
    result_sten_subset.to_netcdf(
        output_file_name, 
        compute=True
    )

print(f"Success: Subset run done. Results saved to '{output_file_name}'")

 Running dask computation on subset...
Success: Subset run done. Results saved to 'gridded_oyster_output_subsetpoint.nc'


In [28]:
# # Validation on subset grid

# Coordinates for Gullmarsfjord, Sweden
validation_lat = 58.2565
validation_lon = 11.472143

# Define a wider 1.0 x 1.0 degree box.
lat_slice = slice(validation_lat - 0.5, validation_lat + 0.5)
lon_slice = slice(validation_lon - 0.5, validation_lon + 0.5)

# Run the check function
check_subset_validity(
    ds_list=[ds_poc, ds_sal], 
    var_names=[poc_var_name, salinity_var_name], 
    lat=validation_lat, 
    lon=validation_lon
)


Checking Validity for Lat: 58.2565, Lon: 11.472143
Nearest grid point found at: Lat=58.38, Lon=11.38
⚠️WARNING: poc is ALL NaN at this location. No model run possible.
✅ sos contains valid data.
❌FAILED: One or more critical variables are missing data. Try a new location.


In [29]:
# # Validation on subset grid

# # Coordinates for Gullmarsfjord, Sweden

validation_lat = 58.250225
validation_lon = 11.452745

# validation_lat = 45.125 
# validation_lon = -7.675

# Define a wider 1.0 x 1.0 degree box.
lat_slice = slice(validation_lat - 0.5, validation_lat + 0.5)
lon_slice = slice(validation_lon - 0.5, validation_lon + 0.5)

# Run the check function


status, valid_lat, valid_lon = check_subset_validity_return_nearest(
    ds_list=[ds_poc, ds_sal], 
    var_names=[poc_var_name, salinity_var_name], 
    lat=validation_lat, 
    lon=validation_lon,
    # 2.0  # Search within 2 degrees (~200 km)
    search_radius_deg=6
)

if status == 'passed':
    print(f"✅ Use these coordinates: Lat={valid_lat:.4f}, Lon={valid_lon:.4f}")
    # Update your slices with the valid coordinates
    lat_slice = slice(valid_lat - 0.5, valid_lat + 0.5)
    lon_slice = slice(valid_lon - 0.5, valid_lon + 0.5)
else:
    print("\n❌ Could not find a valid location. Try a different region.")


print("")
print(status, valid_lat, valid_lon)

Checking Validity for Lat: 58.250225, Lon: 11.452745
  ❌ poc is ALL NaN at this location
  ✅ sos contains valid data
Nearest grid point to request: Lat=58.38, Lon=11.38
⚠️ Requested point has missing data for: poc
   Searching for nearest valid point...
   Searching 48 x 48 = 2304 grid points...
   Checked 2304 points in search area
❌FAILED: No valid point found within 6 degrees of requested location.
  All 2304 points checked had missing data for one or more variables:
  Missing variables at requested point: poc

❌ Could not find a valid location. Try a different region.

None None None


In [30]:

print(f"Creating subset for lat: {lat_slice}, lon: {lon_slice}")
ds_subset = ds_daily.sel(
    latitude=lat_slice, 
    longitude=lon_slice
)

print("Loading small subset into memory...")
ds_subset.load()

print("Validation subset ready")
print(ds_subset)

Creating subset for lat: slice(57.750225, 58.750225, None), lon: slice(10.952745, 11.952745, None)
Loading small subset into memory...
Validation subset ready
<xarray.Dataset> Size: 47kB
Dimensions:      (time: 60, latitude: 4, longitude: 4)
Coordinates:
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
  * latitude     (latitude) float32 16B 57.88 58.12 58.38 58.62
  * longitude    (longitude) float32 16B 11.12 11.38 11.62 11.88
    depth        float32 4B 0.0
Data variables:
    salinity     (time, latitude, longitude) float64 8kB 39.33 42.18 ... nan nan
    POC          (time, latitude, longitude) float64 8kB nan nan nan ... nan nan
    temperature  (time, latitude, longitude) float64 8kB -8.405 4.195 ... 7.106
    Chl          (time, latitude, longitude) float64 8kB -9.847 ... -0.638
    POM          (time, latitude, longitude) float64 8kB 8.978 -4.89 ... 0.6174
    TPM          (time, latitude, longitude) float64 8kB -2.643 13.0 ... -4.985


In [31]:
# Run apply_ufunc on  Subset
result_sten_subset = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # ---  SUBSET data ---
    ds_subset['temperature'],   
    ds_subset['salinity'],
    ds_subset['Chl'],
    ds_subset['POC'],
    ds_subset['POM'],
    ds_subset['TPM'],

    input_core_dims=[['time']] * 6,
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    dask='parallelized',
    output_dtypes=[float],
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)


# Add time coordinate back and set attributes
result_sten_subset = result_sten_subset.assign_coords(time=time_horizon)
result_sten_subset = result_sten_subset.rename('soft_tissue_energy')
result_sten_subset.attrs['units'] = 'J'
result_sten_subset.attrs['long_name'] = 'Oyster Soft Tissue Energy'

print("Task graph built. Ready to compute subset.")
print(result_sten_subset)

TypeError: run_fabm_at_point_full() missing 1 required positional argument: 'log_filename'

In [ ]:

# Compute with progress bar
print(" Running dask computation on subset...")

output_file_name = "gridded_oyster_output_GULLMARSFJORD.nc"

with ProgressBar():
    # Save to a new file
    result_sten_subset.to_netcdf(
        output_file_name, 
        compute=True
    )

print(f"Success: Subset run done. Results saved to '{output_file_name}'")